I am creading this notebook so i can carry out research on pairs trading; especially the chose of best pairs.

This notebook is designed for pure random research. Its structure is not definite and changes with the flow of the current research to suppliment a pairs trading strategy.

In [154]:
import yfinance as yf
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import datetime

In [155]:
# Insert ticker and time frame to investigate
# the goal is to run the ADF(reject null hyp) and once that passes getting the relevant lookback window
# So far we know correlation does not transalate well to a good pair

tickers = ['KO', 'KDP']
n = 180/365
start_time = datetime.datetime.now() - datetime.timedelta(days=365*n)
end_time = datetime.datetime.now()

In [156]:
# using yfinance
data = yf.download(tickers=tickers, start=start_time, end=end_time)
data1 = data.Close

/var/folders/q1/n1z4xy556cjbxzn0zg0twy3m0000gn/T/ipykernel_2171/3562687133.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers=tickers, start=start_time, end=end_time)
[*********************100%***********************]  2 of 2 completed


I want to start by checking the correlation matrix of these and then check the Augmented DickerFueller test. The goal is to see if the highest correlated assets do possess a unit root or not.

In [157]:
from statistics import correlation
import seaborn as sns
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm

# Use a Statistic OLS Regression

In [158]:
corr_matrix = data1.corr()

In [159]:
unstacked = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        ).unstack()

highest_pairs = unstacked.abs().sort_values(ascending=False)
win_pair = highest_pairs.index[0]

ticker_a = win_pair[0]
ticker_b = win_pair[1]

df = pd.DataFrame(index = data1.index)

df[ticker_a] = data1[ticker_a]
df[ticker_b] = data1[ticker_b]

print(f'The pair being used for analysis includes {ticker_a} and {ticker_b}.')

The pair being used for analysis includes KO and KDP.


In [160]:
df = df.dropna()

In [161]:
# perform static OLS

X = df[ticker_a].values.reshape(-1, 1)
X = sm.add_constant(X)
y = df[ticker_b].values.reshape(-1, 1)
model = LinearRegression()
model.fit(X, y)

# get the residuals
residuals = y - model.predict(X)

In [ ]:
ts_dataframe = pd.DataFrame({"Value": residuals.flatten()})

# run the Augmented Dickey-Fuller test
result = adfuller(ts_dataframe["Value"].values)

# extract and print the individual metrics
print("--- ADF Test Results ---")
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")
print(f"Lags Used: {result[2]}")
print(f"Number of Observations: {result[3]}")
print("Critical Values:")
for key, value in result[4].items():
    print(f"   {key}: {value:.4f}")

# corrected statistical interpretation
p_value = result[1]
adf_statistic = result[0]
critical_value_5pct = result[4]["5%"]

print("\n--- Interpretation ---")

# check if p-value is below 5% AND ADF statistic is more negative than the critical value
if p_value <= 0.05 and adf_statistic < critical_value_5pct:
    print(f"Reject the Null Hypothesis (H0). The p-value ({p_value:.4f}) is below 0.05.")
    print("Conclusion: The time series is STATIONARY and mean-reverting. (TRADABLE)")
else:
    print(f"Fail to reject the Null Hypothesis (H0).")
    print("Conclusion: The time series is NON-STATIONARY. The spread is a random walk. (UNTRADABLE)")

--- ADF Test Results ---
ADF Statistic: -3.0730
p-value: 0.0286
Lags Used: 0
Number of Observations: 122
Critical Values:
   1%: -3.4851
   5%: -2.8855
   10%: -2.5796

--- Interpretation ---
Reject the Null Hypothesis (H0). The p-value (0.0286) is below 0.05.
Conclusion: The time series is STATIONARY and mean-reverting. (TRADABLE)


# Using Cointegration

In [ ]:
from statsmodels.tsa.stattools import coint

# verify using cointegration 
ts_dataframe = pd.DataFrame({"Asset_X": data1[ticker_a], "Asset_Y": data1[ticker_b]})
ts_dataframe = ts_dataframe.dropna()
score, p_value, crit_value = coint(ts_dataframe["Asset_Y"], ts_dataframe["Asset_X"])

print(f"Cointegration Test Statistic: {score:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"5% Critical Value: {crit_value[1]:.4f}")


Cointegration Test Statistic: -3.0859
p-value: 0.0912
5% Critical Value: -3.3867


# Rolling OLS

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS

df_1 = df.copy()
df = pd.DataFrame({"Asset_X": df_1[ticker_a], "Asset_Y": df_1[ticker_b]})

# define the independent variable (X) and add a constant for Alpha (α)
X = sm.add_constant(df["Asset_X"])
Y = df["Asset_Y"]

# fit the Rolling OLS model
# 'window=60' means it looks at the trailing 60 days to calculate today's Beta and Alpha
window_size = 21
rolling_model = RollingOLS(Y, X, window=window_size)
rolling_results = rolling_model.fit()

# extract the rolling parameters (Alpha (Spread) and Beta)
# .params returns a DataFrame with 'const' (Alpha) and 'Asset_X' (Beta) columns
rolling_params = rolling_results.params
df["Rolling_Alpha(Spread)"] = rolling_params["const"]
df["Rolling_Beta"] = rolling_params["Asset_X"]

# calculate the Rolling Residuals (The Spread)
# formula: e_t = Y_t - (Beta_t * X_t + Alpha_t)
df["Rolling_Residuals"] = (
    df["Asset_Y"] - (df["Rolling_Beta"] * df["Asset_X"] + df["Rolling_Alpha(Spread)"])
)

# drop the first 59 days (they will be NaN because the window isn't full yet)



In [165]:
# calculate rolling mean, std
df['Mean'] = df['Rolling_Alpha(Spread)'].rolling(window=window_size).mean()
df['Std'] = df['Rolling_Alpha(Spread)'].rolling(window=window_size).std()

# Half - Life of Mean Revertion -> Calculating the ideal Look-Back Window

In order to have the optimal lookback window we typically calculate the half-life of mean reversion. This is the time it takes for the price spread to revert to its historical mean. A common quantitative approach is setting the moving window size to 3 or 4 times the calculated half-life of the pair's spread. For daily data, standard choices cluster around 30 days or 60-120 days. Excessively short windows create false signals from market noise, while excessively long windows mask structural shifts and break down when market regimes change.

## The math
1. Formulate the regression

Regress the daily change in the spread against the previous day's soread value:
$$\Delta S_t = \lambda S_{t-1} + \mu + \epsilon_t,$$

- $\Delta S_t :$ The change in the spread from yesterday to today ($S_t - S_{t-1}$)
- $S_{t-1} :$ Yesterday's spread value,
- $\lambda :$ The regression coefficient representing the speed of mean reversion.
- $\mu :$ A constant (intercept)
- $\epsilon_t :$ The error term (residual).

2. Apply the half life formula

Once you run the linear regression and find the coefficient $\lambda$, calculate the half-life (HL) using ln:
$$HL = - \frac{ln(2)}{\lambda}$$

3. Interpret the result

- Negative $\lambda$: indicates the spread is mean reverting. If $\lambda = -0.1$ the spread closes $10\%$ if its gap to the mean each day.
- Half-Life Value: The resulting number is the expected number of days (or bars) it takes for the spread to close half of its deviation from the mean.
- Positive $\lambda$: The spread is diverging meaning pairs trading will not work.

In [ ]:
# lets apply half-life

df = df.dropna()
delta_S = np.diff(df['Rolling_Alpha(Spread)'])
lagged_S = df['Rolling_Alpha(Spread)'][:-1]


lagged_S_with_constant = sm.add_constant(lagged_S)
model = sm.OLS(delta_S, lagged_S_with_constant).fit()
lambda_coef = model.params.iloc[1]

In [ ]:
# calculate half-life
half_life = -np.log(2) / lambda_coef
print(f"Half-life of mean reversion {half_life:.2f} days with lambda value of {lambda_coef}. Ideal lookback window is {half_life:.0f}.")

Half-life of mean reversion 14.61 days with lambda value of -0.04743212328090155. Ideal lookback window is 15.


For this phase we did a little data snooping aka lookforward bias. This was seen when we determined the optimal z-score. 

Going forward we need to determine Training period so we can :

Use historical data to determine:

* Pair selection
* Hedge ratio methodology
* Z-score threshold
* Entry/exit rules